# Trashpanda: full training and provenance
Choose **Llama** or **GPT-2** for a small synthetic walkthrough. Check **full_run** for the existing **298M-parameter Llama / 6,000,000,000-target** training recipe with real FineWeb-Edu data and the pinned 32K tokenizer. The full-run checkbox overrides the demo token budget and checkpoint interval, and continues automatically across evaluation intervals until the budget or a stopping criterion is reached. Full-size GPT-2 is tracked in [#61](https://github.com/rawkintrevo/trashpanda/issues/61); selecting it fails explicitly instead of training a tiny substitute.

The unchecked demo uses randomly initialized tiny models and a 32-token synthetic vocabulary; it needs only a CPU. The full recipe needs a CUDA/BF16 GPU (designed for A100 80GB), substantial Drive capacity, package/tokenizer/data downloads, and potentially multiple runtime sessions. This notebook has not yet been validated on a live GPU/Drive runtime: [#64](https://github.com/rawkintrevo/trashpanda/issues/64).

**After a timeout:** reconnect, run from the top, authorize Drive again, and use the same run name and settings. Checkpoints include model, optimizer, scheduler, sampler and RNG state; the validation history and stopping decision also live in Drive. Use a **new run name** to change mode, architecture, budget or stopping policy, including when upgrading from the original notebook.

Only run one writer per run folder. Work after the last remotely persisted checkpoint can be lost. Drive mounting does not guarantee buffered writes reached remote storage; see [Colab runtime and Drive limits](https://research.google.com/colaboratory/faq.html). Earlier checkpoints are retained; automatic verified retention is tracked in [#62](https://github.com/rawkintrevo/trashpanda/issues/62).


In [ ]:
#@title 1. Connect your Google Drive (authorize the prompt)
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/trashpanda")
if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("Connect Google Drive before continuing.")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Persistent storage:", DRIVE_ROOT)

In [ ]:
#@title 2. Select a model and persistent run
full_run = False #@param {type:"boolean"}
model_architecture = "llama" #@param ["llama", "gpt2"]
run_name = "my-first-run" #@param {type:"string"}
token_budget = 12400 #@param {type:"integer"}
steps_per_session = 25 #@param {type:"integer"}
checkpoint_every = 5 #@param {type:"integer"}
early_stopping = True #@param {type:"boolean"}
validation_patience = 5 #@param {type:"integer"}
validation_min_delta = 0.01 #@param {type:"number"}
target_validation_loss = 0.0 #@param {type:"number"}
full_validation_interval = 500 #@param {type:"integer"}
full_train_documents = 30000 #@param {type:"integer"}
full_validation_documents = 1000 #@param {type:"integer"}

import json
import math
import os
import re
import uuid

if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,79}", run_name):
    raise ValueError("Use a run name containing only letters, numbers, underscores or hyphens.")
if model_architecture not in {"llama", "gpt2"}:
    raise ValueError("Select llama or gpt2.")
if any(type(v) is not int or v <= 0 for v in (token_budget, steps_per_session, checkpoint_every)):
    raise ValueError("Budget, session steps, and checkpoint interval must be positive integers.")
if type(validation_patience) is not int or validation_patience < 1:
    raise ValueError("Validation patience must be a positive integer.")
if any(not math.isfinite(v) or v < 0 for v in (validation_min_delta, target_validation_loss)):
    raise ValueError("Validation thresholds must be finite and nonnegative.")
if any(type(v) is not int or v < 1 for v in (full_validation_interval, full_train_documents, full_validation_documents)):
    raise ValueError("Full-run intervals and document counts must be positive integers.")
if full_run and model_architecture == "gpt2":
    raise ValueError("Full-size GPT-2 is not implemented yet: https://github.com/rawkintrevo/trashpanda/issues/61. "
                     "Use full-run Llama or uncheck full_run for the tiny GPT-2 demo.")
recipe_name = "300m-a100" if full_run else "tiny"
# Full run overrides the demonstration budget and checkpoint interval.
token_budget = 6_000_000_000 if full_run else token_budget
checkpoint_every = 500 if full_run else checkpoint_every
updates_per_evaluation = full_validation_interval if full_run else steps_per_session
RUN = DRIVE_ROOT / run_name
RUN.mkdir(parents=True, exist_ok=True)

def write_json(path, value):
    temporary = path.with_name(path.name + ".tmp-" + uuid.uuid4().hex)
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(value, handle, indent=2, sort_keys=True)
        handle.flush()
        os.fsync(handle.fileno())
    temporary.replace(path)

# Pin the demonstrated implementation across runtime restarts.
CODE_REVISION = "acc418fc9e41e42d885dca1dc8baf9aa075d4290"
settings = {"architecture": model_architecture, "token_budget": token_budget,
            "checkpoint_interval": checkpoint_every, "code_revision": CODE_REVISION,
            "full_run": full_run, "recipe": recipe_name,
            "full_data": {"train_documents": full_train_documents,
                          "validation_documents": full_validation_documents} if full_run else None,
            "full_validation_interval": full_validation_interval if full_run else None,
            "stopping": {"enabled": early_stopping, "patience": validation_patience,
                         "min_delta": validation_min_delta, "target_loss": target_validation_loss}}
settings_path = RUN / "settings.json"
if settings_path.exists():
    if json.loads(settings_path.read_text()) != settings:
        raise ValueError("This run has different saved settings. Restore them or choose a new run name. "
                         + settings_path.read_text())
else:
    write_json(settings_path, settings)
print("Run folder:", RUN)
print("Settings:", settings)
print("Mode:", recipe_name, "| requested targets:", token_budget)


In [ ]:
#@title 3. Install the pinned source revision
import subprocess
import sys

if sys.version_info < (3, 11):
    raise RuntimeError("Use a Colab runtime with Python 3.11 or newer.")
REPO = Path("/content/trashpanda-source")
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "https://github.com/rawkintrevo/trashpanda.git", str(REPO)], check=True)
subprocess.run(["git", "fetch", "origin", settings["code_revision"]], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--detach", settings["code_revision"]], cwd=REPO, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO) + ("[datasets]" if full_run else "")], check=True)
for flag in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE"):
    os.environ.pop(flag, None)
# Record each runtime's dependencies rather than overwriting earlier sessions.
versions = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(RUN / ("environment-" + uuid.uuid4().hex + ".txt")).write_text(versions)
print("Installed revision:", settings["code_revision"])

## Persist inputs and preparation
The full mode resolves and saves an immutable FineWeb-Edu revision, downloads sequential non-overlapping train/validation slices, retains their JSONL sources in Drive, and prepares shards plus a tokenizer snapshot. Completed inputs/preparation are reused. Interrupted downloads/preparation currently restart their stage; resumable acquisition is tracked in [#63](https://github.com/rawkintrevo/trashpanda/issues/63).

The default sample is **30,000 train / 1,000 validation documents**, much smaller than 6B unique tokens. Training repeats that sample; the cell prints its target count and the implied repeat factor. Increase the document count for your data plan, with corresponding time/storage costs. Source identities and hashes are evidence, not assertions of dataset approval.

Multi-GB full checkpoints accumulate on Drive. At 500-update boundaries, a 6B run has about 46 checkpoints (more with shorter evaluation intervals), potentially hundreds of GB including optimizer state, data and exports. Verified retention/capacity planning is an open prerequisite for unattended long runs: [#62](https://github.com/rawkintrevo/trashpanda/issues/62).


In [ ]:
#@title 4. Prepare persistent synthetic inputs and configuration
from dataclasses import asdict, replace
import torch
from trashpanda.config import model_recipe, config_from_dict
from trashpanda.prepare import prepare_splits
from trashpanda.tokenizer import TinyWordLevelTokenizer
from trashpanda.data import SequenceReader

# Limit CPU overhead for these very small models.
torch.set_num_threads(2)
base = model_recipe(recipe_name, model_architecture)
if full_run:
    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
        raise RuntimeError("The full recipe requires a CUDA GPU with BF16 support; select an A100 runtime.")
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory GiB:", torch.cuda.get_device_properties(0).total_memory / 2**30)
    print("The unchanged recipe was designed for A100 80GB; smaller devices may run out of memory. See #64.")

SOURCES = RUN / "sources"
SOURCES.mkdir(exist_ok=True)
if full_run:
    from datasets import load_dataset
    from huggingface_hub import HfApi
    from trashpanda.tokenizer import load_tokenizer
    # Persist the immutable source revision before downloading any records.
    source_spec_path = RUN / "data-source.json"
    if not source_spec_path.exists():
        source_revision = HfApi().dataset_info("HuggingFaceFW/fineweb-edu").sha
        if not source_revision:
            raise RuntimeError("Could not resolve an immutable FineWeb-Edu revision.")
        write_json(source_spec_path, {"dataset": "HuggingFaceFW/fineweb-edu",
                                    "subset": "sample-10BT", "revision": source_revision})
    source_spec = json.loads(source_spec_path.read_text())
    source_set = SOURCES / "fineweb"
    if not source_set.exists():
        staging_sources = SOURCES / ("fineweb.partial-" + uuid.uuid4().hex)
        staging_sources.mkdir()
        stream = iter(load_dataset(source_spec["dataset"], name=source_spec["subset"],
                                   revision=source_spec["revision"], split="train", streaming=True))
        for split, count in (("train", full_train_documents), ("validation", full_validation_documents)):
            with (staging_sources / (split + ".jsonl")).open("w", encoding="utf-8") as handle:
                for _ in range(count):
                    record = next(stream, None)
                    if not record or not isinstance(record.get("text"), str) or not record["text"]:
                        raise ValueError("Stream ended or supplied an invalid text record.")
                    handle.write(json.dumps({"text": record["text"]}, ensure_ascii=False) + "\n")
                handle.flush()
                os.fsync(handle.fileno())
        staging_sources.rename(source_set)
    source_descriptors = [
        {"source_id": f"hf:{source_spec['dataset']}:{source_spec['revision']}:{split}",
         "source": source_spec["dataset"], "source_revision": source_spec["revision"],
         "selection": {"subset": source_spec["subset"], "split": "train",
                       "start_document": offset, "documents": count}, "evidence_state": "observed"}
        for split, offset, count in (("train", 0, full_train_documents),
                                     ("validation", full_train_documents, full_validation_documents))]
else:
    source_set = SOURCES
    for split, documents, words, offset in (("train", 8, 128, 0), ("validation", 4, 96, 13)):
        path = SOURCES / (split + ".jsonl")
        expected = "".join(json.dumps({"text": " ".join(f"w{(i + d + offset) % 29}" for i in range(words))}) + "\n"
                           for d in range(documents))
        if path.exists() and path.read_text() != expected:
            raise ValueError(f"Source changed or incomplete: {path}. Use a new run name or restore the file.")
        if not path.exists():
            temporary = path.with_suffix(".partial-" + uuid.uuid4().hex)
            temporary.write_text(expected)
            temporary.replace(path)
    source_descriptors = [{"source_id": "synthetic:" + split, "source_revision": "fixture-1",
                           "evidence_state": "observed"} for split in ("train", "validation")]

PREPARED = RUN / "prepared"
if not PREPARED.exists():
    tokenizer = (load_tokenizer(base.data.tokenizer_name, revision=base.data.tokenizer_revision,
                               vocab_size=base.model.vocab_size, local_files_only=False)
                 if full_run else TinyWordLevelTokenizer())
    staging = RUN / ("prepared.partial-" + uuid.uuid4().hex)
    prepare_splits(staging, train_path=source_set / "train.jsonl",
                   validation_path=source_set / "validation.jsonl", tokenizer=tokenizer,
                   context_length=base.model.context_length, vocab_size=base.model.vocab_size,
                   source_descriptors=source_descriptors)
    staging.rename(PREPARED)
from trashpanda.cli import _tokenizer_from_prepared
tokenizer = _tokenizer_from_prepared(PREPARED, base)
requested_config = replace(base,
    training=replace(base.training, token_budget=token_budget,
                     checkpoint_interval=checkpoint_every, device=base.training.device if full_run else "auto"),
    runtime=replace(base.runtime, output_dir=str(RUN / "checkpoints")))
config_path = RUN / "config.json"
if config_path.exists():
    config = config_from_dict(json.loads(config_path.read_text()))
    if config.to_dict() != requested_config.to_dict():
        raise ValueError("Saved configuration differs; use a new run name.")
else:
    config = requested_config
    write_json(config_path, config.to_dict())
dataset = SequenceReader(PREPARED, split="train", verify_hash=True)
print(json.dumps(config.to_dict(), indent=2))
print("Prepared manifest:", PREPARED / "manifest.json")
prepared_targets = len(dataset) * (config.model.context_length - 1)
print("Prepared training targets:", prepared_targets)
print("Requested target budget / prepared targets:", round(token_budget / prepared_targets, 2))
print("Training repeats prepared sequences; the budget is NOT a count of unique source tokens.")


## Training and stopping metrics
Unchecked demo mode trains `steps_per_session` additional updates and returns. Full-run mode uses `full_validation_interval` updates per evaluation, continuing automatically until it stops or the runtime disconnects. Full checkpoints are saved every 500 updates and at evaluation boundaries; the requested 6B targets are rounded up to a whole optimizer update.

Validation reports held-out loss, perplexity, and the number of evaluated targets (first 64 validation batches for full mode; 8 for tiny). With `early_stopping` enabled, stop when loss reaches `target_validation_loss` (0 disables this target), or when loss has not improved by at least `validation_min_delta` for `validation_patience` evaluations. These configurable defaults are illustrative, not validated quality thresholds. Nonfinite validation loss always stops. The full token budget is a ceiling, not a promise of convergence.

`stopping.json` preserves observations, the best checkpoint, patience counter, and the stop reason. Reconnecting does not reset patience or count the same checkpoint twice. Exports below use the **latest checkpoint**, not automatically the best one. Core CLI training still stops by budget/steps/signals; this validation controller is notebook-specific.

Checkpoint recovery scans newest first, skips unreadable files, and stops if none are usable. Observations newer than a recovered checkpoint are discarded. Interrupted preparation/export folders are not treated as completed artifacts. Package versions are saved for each runtime; dependency or hardware changes can affect numerical reproducibility.


In [ ]:
#@title Validation stopping policy
# Validation stopping uses held-out loss (lower is better).
# Patience counts evaluations, not optimizer updates or runtime sessions.
import math

def stopping_decision(observations, *, enabled, patience, min_delta, target_loss):
    if patience < 1 or min_delta < 0 or target_loss < 0:
        raise ValueError("Invalid stopping criteria")
    reference_loss = None
    best_loss = None
    best_checkpoint = None
    stale_evaluations = 0
    reason = None
    for observation in observations:
        loss = observation["loss"]
        if loss is None or not math.isfinite(loss):
            reason = "non_finite_validation_loss"
            break
        if reference_loss is None or loss < reference_loss - min_delta:
            reference_loss = loss
            stale_evaluations = 0
        else:
            stale_evaluations += 1
        if best_loss is None or loss < best_loss:
            best_loss = loss
            best_checkpoint = observation["checkpoint"]
        if enabled and target_loss > 0 and loss <= target_loss:
            reason = "target_validation_loss"
            break
        if enabled and stale_evaluations >= patience:
            reason = "validation_plateau"
            break
    return {"best_loss": best_loss, "best_checkpoint": best_checkpoint,
            "stale_evaluations": stale_evaluations, "stop_reason": reason}


In [ ]:
#@title 5. Train or resume (full run or demo chunk)
from trashpanda.checkpoint import list_checkpoints, load_checkpoint
from trashpanda.model import build_model
from trashpanda.train import train

CHECKPOINTS = RUN / "checkpoints"

def latest_usable_checkpoint():
    candidates = list_checkpoints(CHECKPOINTS)
    probe = build_model(config.model, seed=config.training.seed) if candidates else None
    for path in reversed(candidates):
        try:
            payload = load_checkpoint(path, model=probe)
            # Ensure this has the state required for a training restart.
            for key in ("state", "optimizer", "sampler", "rng", "config"):
                if key not in payload:
                    raise ValueError(f"Missing checkpoint field: {key}")
        except Exception as exc:
            print(f"Skipping unreadable checkpoint {path.name}: {exc}")
            continue
        if payload["config"]["training"] != config.to_dict()["training"]:
            raise ValueError("Checkpoint training settings differ from this run.")
        if payload["config"]["model"] != config.to_dict()["model"]:
            raise ValueError("Checkpoint architecture differs from this run.")
        if payload.get("data_identity") != dataset.manifest.get("identity"):
            raise ValueError("Checkpoint data identity differs from prepared data.")
        return path, payload["state"]
    if candidates:
        raise RuntimeError("No usable checkpoint. Restore a backup or choose a new run name.")
    return None, {"update_step": 0, "consumed_tokens": 0}

from contextlib import nullcontext
from trashpanda.evaluate import evaluate
from trashpanda.provenance import sha256_file

checkpoint, state = latest_usable_checkpoint()
VALIDATION = SequenceReader(PREPARED, split="validation")
STOPPING_FILE = RUN / "stopping.json"
history = json.loads(STOPPING_FILE.read_text())["observations"] if STOPPING_FILE.exists() else []
# Discard observations beyond the recovered checkpoint, including a changed replay.
history = [row for row in history if row["step"] <= state["update_step"]]
if checkpoint and history and history[-1]["step"] == state["update_step"]:
    if history[-1]["checkpoint_sha256"] != sha256_file(checkpoint):
        history.pop()

def observe_checkpoint(checkpoint, state, trained_model=None):
    if not checkpoint or (history and history[-1]["step"] == state["update_step"]):
        return
    observed_model = trained_model if trained_model is not None else build_model(config.model)
    if trained_model is None:
        load_checkpoint(checkpoint, model=observed_model)
    try:
        with torch.autocast("cuda", dtype=torch.bfloat16) if full_run else nullcontext():
            observed = evaluate(observed_model, VALIDATION, max_batches=64 if full_run else 8,
                                device="cuda" if full_run else "cpu")
        loss = observed.loss if math.isfinite(observed.loss) else None
        perplexity, validation_targets = observed.perplexity, observed.tokens
    except FloatingPointError as exc:
        print("Validation failed:", exc)
        loss, perplexity, validation_targets = None, None, None
    history.append({"step": state["update_step"], "targets": state["consumed_tokens"],
                    "checkpoint": checkpoint.name, "checkpoint_sha256": sha256_file(checkpoint),
                    "loss": loss, "perplexity": perplexity,
                    "validation_targets": validation_targets})
    print("Validation:", history[-1])

def save_stopping_status():
    status = stopping_decision(history, **settings["stopping"])
    if status["stop_reason"] is None and state["consumed_tokens"] >= config.training.token_budget:
        status["stop_reason"] = "token_budget"
    write_json(STOPPING_FILE, {"observations": history, **status})
    return status

# A timeout between checkpoint save and evaluation is repaired before more training.
observe_checkpoint(checkpoint, state)
status = save_stopping_status()
if status["stop_reason"]:
    print("Stopped:", status["stop_reason"])
while not status["stop_reason"]:
    print("Resuming:" if checkpoint else "Starting:", checkpoint or model_architecture)
    chunk_target = state["update_step"] + updates_per_evaluation
    result = train(config, dataset, tokenizer=tokenizer,
                   resume=checkpoint, checkpoint_dir=CHECKPOINTS,
                   max_steps=chunk_target)
    checkpoint, state = result.checkpoint, result.state.to_dict()
    observe_checkpoint(checkpoint, state, result.model)
    del result
    status = save_stopping_status()
    if not full_run or state["update_step"] < chunk_target:
        break
print("Checkpoint:", checkpoint)
print("Progress:", state)
print("Stopping metrics:", status)
print("Validation history and stop reason:", STOPPING_FILE)
print("Training metrics:", CHECKPOINTS / "metrics.jsonl")


In [ ]:
#@title 6. Evaluate, sample, and export the latest checkpoint with provenance
from trashpanda.evaluate import evaluate, sample_text
from trashpanda.export import export_checkpoint
from trashpanda.provenance import inspect_bundle, verify_bundle, sha256_file

checkpoint, state = latest_usable_checkpoint()
if checkpoint is None:
    raise RuntimeError("Run a training chunk before exporting.")
model = build_model(config.model)
load_checkpoint(checkpoint, model=model)
validation = SequenceReader(PREPARED, split="validation")
with torch.autocast("cuda", dtype=torch.bfloat16) if full_run else nullcontext():
    metrics = evaluate(model, validation, max_batches=64 if full_run else 2,
                       device="cuda" if full_run else "cpu")
print("Evaluation:", metrics)
print("Sample:", sample_text(model, tokenizer, "The future of education is" if full_run else "w0 w1",
                                max_new_tokens=8, seed=42, device="cuda" if full_run else "cpu"))
write_json(RUN / f"evaluation-{state['update_step']:08d}.json", asdict(metrics))

# Include the checkpoint digest so recovered/replayed steps cannot collide.
del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
EXPORTS = RUN / "exports"
EXPORTS.mkdir(exist_ok=True)
EXPORT = EXPORTS / (checkpoint.stem + "-" + sha256_file(checkpoint)[:12])
if not EXPORT.exists():
    staging = EXPORTS / (EXPORT.name + ".partial-" + uuid.uuid4().hex)
    export_checkpoint(checkpoint, staging)
    verification = verify_bundle(staging)
    if not verification.ok:
        raise RuntimeError(verification.to_dict())
    staging.rename(EXPORT)
print("Portable Hugging Face export:", EXPORT)
print(json.dumps(inspect_bundle(EXPORT), indent=2))
print(json.dumps(verify_bundle(EXPORT).to_dict(), indent=2))

The export contains model/tokenizer files, `provenance-manifest.json`, and digest-addressed records in `provenance/records/`. It carries input, preparation and run evidence from the checkpoint. The evaluation above is saved separately; it is not retroactively added to checkpoint evidence.

Verification distinguishes integrity, evidence completeness, signatures, and policy. **Unsigned means unauthenticated**, even when hashes verify. No policy means authorization has not been evaluated. An intermediate export represents only the updates completed so far, not the requested full budget.

In [ ]:
#@title 7. Verify a relocated copy and demonstrate tamper detection
import shutil
import tempfile
from transformers import AutoModelForCausalLM, AutoTokenizer

# Scratch copies only: never modify the original Drive export.
with tempfile.TemporaryDirectory() as temporary:
    relocated = Path(temporary) / "export"
    shutil.copytree(EXPORT, relocated)
    assert verify_bundle(relocated).ok
    restored = AutoModelForCausalLM.from_pretrained(relocated, local_files_only=True, trust_remote_code=False)
    reloaded_tokenizer = AutoTokenizer.from_pretrained(relocated, local_files_only=True, trust_remote_code=False)
    print("Offline reload:", restored.config.model_type, "vocabulary:", reloaded_tokenizer.vocab_size)
    payload = relocated / "config.json"
    with payload.open("a") as handle:
        handle.write("\n ")
    tampered = verify_bundle(relocated)
    assert not tampered.ok
    print("Tampered copy integrity:", tampered.integrity)
print("Original Drive bundle remains valid:", verify_bundle(EXPORT).ok)

## Returning after a runtime expires
1. Open this notebook, reconnect, and run cells 1–5 with the same run name and settings. Drive will prompt for authorization when needed.
2. Check the printed resumed checkpoint and counters. Repeat cell 5 to train more; run cells 6–7 whenever you want an export.
3. Find all durable artifacts under `MyDrive/trashpanda/<run_name>/`. Download the **whole export folder**, including `provenance/`, to carry the evidence with the model.

Keep earlier checkpoints until you have confirmed a later one is usable. Interrupted `.partial-*` artifacts can be removed manually once no runtime is using the run. Drive I/O/quota errors must be resolved before continuing. Metrics may contain replayed steps after recovery; checkpoint counters are the authority for committed progress. Never run two writers against the same folder.

Full-run follow-ups: [production GPT-2 #61](https://github.com/rawkintrevo/trashpanda/issues/61), [verified checkpoint retention #62](https://github.com/rawkintrevo/trashpanda/issues/62), [resumable corpus preparation #63](https://github.com/rawkintrevo/trashpanda/issues/63), and [GPU/cross-runtime validation #64](https://github.com/rawkintrevo/trashpanda/issues/64).

[Trainer-level stopping and portable stop evidence #65](https://github.com/rawkintrevo/trashpanda/issues/65) tracks moving the notebook controller into the shared trainer. Current exports carry training provenance; validation history and stopping decisions are separate Drive files.
